# Lists

**Topic:** `3.1` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Python's ordered, mutable sequence: reference semantics, the two classic list bugs - aliasing and None-returning methods - and the performance profile that follows from an array of pointers.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

## 1. Why This Topic Matters

Every pipeline in the course accumulates data into lists, and the two list bugs strike silently: a shared waypoint plan changes under every component that reads it, and a sorted-assigned-to-None log vanishes one line before it is needed. Mastering reference semantics here is what makes every later collection safe to touch.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain that assignment aliases and that list copies are shallow.
2. Predict which list methods mutate in place and what they return.
3. Copy deliberately at module boundaries with list(), copy() or slicing.
4. Choose between append, extend and concatenation by cost, not habit.
5. Process a list safely when elements must be removed or replaced.
6. State the time complexity of indexing, appending, inserting and scanning.

## 3. Prerequisites

This topic assumes you already have:

- Topic 2.5 range, enumerate and zip

## 4. Mental Model

**Mental model: a list is a row of labeled boxes, and a variable is only a label pointing at the row.** Indexing reads one box. In-place methods write into boxes, so every label pointing at the row sees the change. Assignment of one name to another just prints a second label - it never photographs the row.

| Statement | What it does | Independent copy? |
| --- | --- | --- |
| `b = a` | second label on the same row | no - aliasing |
| `b = a.copy()` | new row, same element references | outer yes, elements no |
| `b = a[:]` / `list(a)` | same as `.copy()` | outer yes, elements no |
| `b = a[i:j]` | new row from a window | yes for the window |
| `a.append(x)` | writes into the row | n/a - mutates in place |
| `b = sorted(a)` | new row in ranking order | yes - `a` is untouched |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| List | An ordered, mutable sequence backed by an array of references. |
| Mutability | The ability of an object to change after it is created. |
| Aliasing | Two or more names bound to the same object. |
| Shallow copy | A new outer container whose elements are the same references. |
| Deep copy | A fully independent clone, including every nested object. |
| In-place | An operation that modifies its target and returns None. |
| Amortised O(1) | Average constant cost per append despite occasional resizing. |
| Sequence protocol | Indexing, slicing, len, membership and iteration on a container. |
| Reference | A name's pointer to an object stored elsewhere in memory. |
| Slice | A window of a sequence that always produces a new list. |

## 6. Conceptual Explanation

A list is Python's general-purpose **ordered, mutable** sequence. Ordered means position is meaningful and stable: `list[0]` is always the first element you put there until you move it. Mutable means the same list object can grow, shrink and change while every name bound to it observes the change. That second property is the whole topic, because it is where both of the classic list bugs come from.

Under the hood a list is an array of pointers: a contiguous block of references to objects that live elsewhere in memory. This is why indexing is O(1) regardless of size, why appending is amortised O(1), and why inserting at the front is O(n) - every later reference must be shifted. It also explains a subtlety beginners miss: a list stores *references*, not values, so `[0] * 3` builds three references to the **same** integer object, and a nested `[row] * n` builds n references to the **same** row.

In [ ]:
waypoints = [(0.0, 0.0), (3.0, 0.0), (3.0, 4.0)]
print('before:', waypoints)

alias = waypoints            # a second name for the SAME list
alias.append((6.0, 4.0))
print('alias appended:', alias)
print('waypoints now :', waypoints)   # changed too - aliasing

Assignment never copies. `alias = waypoints` binds a second name to one list object, so every mutation through either name is visible through both. This is not a defect - sharing is how Python avoids copying large data - but code that *intends* an independent copy must ask for one with `list(waypoints)`, `waypoints[:]` or `waypoints.copy()`, and must understand that all three are **shallow**: the outer list is new, the elements are the same references as before.

## 7. Formal Theory

A list is a sequence supporting the full sequence protocol: indexing `a[i]`, negative indexing `a[-1]`, slicing `a[i:j:k]`, `len(a)`, `a + b`, `a * n`, membership `x in a`, and iteration. Two defining identities keep the mental model honest:

```text
a[i:j] builds a NEW list   |   for names x, y:  x is y  <=>  x and y reference the same object
```

Slicing always allocates. `window = readings[2:5]` copies three references into a fresh list, so later mutations of `readings` do not change `window`, and vice versa. The cost of a slice is O(k) in the number of elements copied - cheap for a window over a log, but a `readings[:]` of a million-element list is a million references moved.

In [ ]:
readings = [0.4, 0.9, 0.7]
window = readings[1:]
window.append(0.2)
print('window:', window)
print('source:', readings)          # the slice was a copy
print('same object?', window is readings)

| Operation | Time | Notes |
| --- | --- | --- |
| a[i] | O(1) | direct pointer read |
| a.append(x) | amortised O(1) | occasional resize, still linear overall |
| a + b / a * n | O(n + m) | builds a new list |
| a.insert(0, x) | O(n) | every reference shifts by one |
| x in a | O(n) | linear scan - use a set for repeated tests |
| a[i:j] | O(j - i) | copies the references in the window |
| del a[i] / a.pop(i) | O(n - i) | cheap at the end, costly at the front |

Mutating methods are the contract students must memorise: `append`, `extend`, `insert`, `remove`, `pop`, `sort`, `reverse` and `clear` all modify the list **in place** and return `None`. The one non-mutating counterpart is the free function `sorted(a)`, which returns a new list and leaves the original untouched. Code that writes `readings = readings.sort()` therefore binds the name to `None` and destroys the list on the next line that uses it.

## 8. Syntax

The canonical forms - literal construction, in-place methods, and the non-mutating alternatives.

In [ ]:
commands = ['stop', 'hold', 'go']

commands.append('log')            # in place, returns None
commands.extend(['rest', 'idle'])  # splice every item of the argument
top = commands.pop()              # remove AND return the last item
commands.insert(0, 'boot')
commands.remove('hold')           # removes the FIRST match by value

print('commands :', commands)
print('popped   :', top)

original = [3, 1, 2]
ordered = sorted(original)        # new list, original untouched
original.sort()                   # in place
print('sorted() :', ordered, ' list.sort():', original)

The anti-pattern - assigning the result of a mutating method, which is always `None`:

```text
readings = [0.4, 0.9, 0.7]
readings = readings.sort()        # None - the list is gone
readings.append(0.1)              # AttributeError: 'NoneType'

readings.sort()                   # correct: call for the side effect
ordered = sorted(readings)        # correct: call for the new list
```

> **append is not extend**
>
> `append(x)` adds x as a single element, even when x is a list; `extend(xs)` splices every element of xs. `a.append([1, 2])` grows the list by one - a nested list - while `a.extend([1, 2])` grows it by two.

## 9. Basic Examples

**Example 1 - build and read a waypoint list.**

In [ ]:
waypoints = []
for x, y in ((0, 0), (3, 0), (3, 4)):
    waypoints.append((x, y))

print('first :', waypoints[0])
print('last  :', waypoints[-1])
print('count :', len(waypoints))
print('slice :', waypoints[1:])

**Example 2 - a running log with membership tests.**

In [ ]:
log = []
for value in (0.4, 0.9, 0.7):
    log.append(round(value, 2))

print('log contains 0.9?', 0.9 in log)
print('first index of 0.7:', log.index(0.7))
print('total readings:', len(log))

**Example 3 - unpacking, which reads better than indexing.** The star pattern captures "everything else" without a slice argument.

In [ ]:
first, *rest, last = [10, 20, 30, 40]
print('first:', first, ' rest:', rest, ' last:', last)

head, tail = commands[:1], commands[1:]
print('head:', head, ' tail:', tail)

## 10. Intermediate Examples

**Windows over a log.** Slicing by computed bounds is the readable way to walk a fixed-width window; the indices come from `range`, so the window size and the stop value stay in one place.

In [ ]:
readings = [0.4, 0.9, 0.7, 0.2, 0.8, 0.5]
size = 3
windows = [readings[i:i + size] for i in range(len(readings) - size + 1)]
print('windows:', windows)
print('last window:', windows[-1], ' is a copy?', windows[-1] is readings)

**Sorting with intent.** `list.sort()` mutates; `sorted()` returns a new list; `reverse=True` inverts; a `key` extracts the ranking criterion so the data itself is never rewritten to be sortable.

In [ ]:
telemetry = [('alpha', 0.42), ('beta', 0.19), ('gamma', 0.75)]

by_value = sorted(telemetry, key=lambda row: row[1], reverse=True)
print('ranked :', by_value)
print('source :', telemetry)          # unchanged by sorted()

**Copying deliberately.** When a function must not disturb the caller's list, copy at the boundary - and remember the copy is shallow.

In [ ]:
def clamp(readings, limit=1.0):
    data = list(readings)             # own the copy
    data = [min(value, limit) for value in data]
    return data

source = [0.4, 1.3, 0.7]
safe = clamp(source)
print('safe  :', safe)
print('source:', source)              # caller's list untouched

## 11. Advanced Examples

**Aliasing inside nested structures.** `[row] * n` repeats the *same* reference n times, so writing to one row writes to all of them. The fix is a comprehension that builds each row fresh.

In [ ]:
broken = [[0] * 3] * 3
broken[0][1] = 7
print('broken:', broken)              # every row changed

grid = [[0] * 3 for _ in range(3)]
grid[0][1] = 7
print('grid  :', grid)                # only row 0 changed

> **shallow copies share their elements**
>
> `a = [[1], [2]]; b = a.copy()` gives two outer lists that still point at the same inner lists. Mutating `b[0]` changes `a[0]`. For independent nested data use `copy.deepcopy(a)` - and pay its cost deliberately.

**Mutation during iteration** silently skips elements, because the loop reads by position while the list shifts underneath it. Iterate a copy (or build a new list) instead.

In [ ]:
targets = [1, 2, 3, 4, 5]

naive = list(targets)
for value in naive:
    if value % 2 == 0:
        naive.remove(value)
print('naive filter:', naive)         # 2 and 4 still present

filtered = [v for v in targets if v % 2 != 0]
print('clean filter:', filtered)

> **identity tools**
>
> `x is y` asks whether two names share one object; `id(x)` prints that object's identity. Use them to prove aliasing in tests: `assert a is not b` is the cheapest guard against an accidental shared reference.

## 12. Code Walkthrough

**A telemetry cleaner that refuses to damage its input.** This is the shape every list-processing function should have: copy first, transform into a new list, return it - the caller's data is never a side effect.

In [ ]:
def clean_readings(raw, low=0.0, high=1.0):
    """Clamp readings into [low, high] and drop non-numeric entries.

    Returns a new list; the input is never modified.
    """
    cleaned = []
    for value in raw:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            continue                       # skip junk before clamping
        cleaned.append(min(max(value, low), high))
    return cleaned

raw = [0.4, 1.3, 'junk', -0.2, 0.7]
print('cleaned:', clean_readings(raw))
print('raw    :', raw)                # identical to before the call

- `cleaned = []` gives the function its own list, so no caller state is at risk.
- The type guard runs before the clamp, so junk never reaches `min`/`max` and never raises.
- `min(max(v, low), high)` clamps in one expression - readable once you name it.
- Returning the new list makes the function composable: the result can be chained, stored or discarded without touching `raw`.

## 13. Line-by-Line Explanation

1. `def clean_readings(raw, low=0.0, high=1.0):` - `raw` is bound to the caller's list; nothing here copies it yet.
2. `cleaned = []` - the function now owns a list no one else can see.
3. `for value in raw:` - iteration reads `raw` by position and never mutates it, so the loop is safe.
4. `if isinstance(value, bool) ...: continue` - `bool` subclasses `int`, so the explicit bool check comes first or `True` would clamp to 1.
5. `cleaned.append(...)` - in-place growth of the private list; `append` returns `None`, so it must stand alone as a statement.
6. `return cleaned` - hand back the new list; `raw` has not been touched at any point, which the final print proves.

## 14. Common Mistakes

**Mistake 1 - assuming assignment copies a list.**

```text
a = [1, 2, 3]
b = a                 # same object
b.append(4)
print(a)              # [1, 2, 3, 4] - a changed too

b = a.copy()          # independent outer list
b.append(5)
print(a)              # [1, 2, 3, 4] - now stable
```

**Mistake 2 - assigning a mutating method's return value.**

```text
readings = [0.4, 0.9]
readings = readings.sort()     # None
readings = readings.reverse()  # would fail the same way
```

**Mistake 3 - append when you meant extend (or the reverse).**

```text
parts = ['ro']
parts.append('ver')            # ['ro', 'ver'] - two elements
parts.extend(['-', '01'])      # ['ro', 'ver', '-', '01'] - spliced
```

**Mistake 4 - `[x] * n` for nested or mutable elements.**

```text
row = [[0]] * 2
row[0].append(1)
print(row)        # [[0, 1], [0, 1]] - ONE inner list, two names

row2 = [[0] for _ in range(2)]   # two independent inner lists
```

## 15. Debugging Techniques

When a list changes that you never touched, print identity, not value. `id()` plus a membership check on `is` shows instantly whether two names share one object.

In [ ]:
a = [(0, 0)]
b = a
c = list(a)

print('a id:', id(a))
print('b id:', id(b), ' shares with a?', b is a)
print('c id:', id(c), ' shares with a?', c is a)

b.append((1, 1))
print('after b.append -> a:', a)
print('after b.append -> c:', c)      # c stayed independent

When a loop drops the wrong elements, the list is probably moving under the iteration. Print the list each pass and the skipped elements become obvious.

In [ ]:
targets = [1, 2, 3, 4]
for pass_no, value in enumerate(list(targets)):
    print(f'pass {pass_no}: seeing {value}, list is {targets}')
    if value % 2 == 0:
        targets.remove(value)
print('result:', targets)             # 4 was never visited

> **`None` returned from a list method is a fingerprint**
>
> If a variable that should hold a list is `None`, search for an assignment from `.sort()`, `.reverse()` or `.pop()` - one of them wrote `None` into the name.

## 16. Best Practices

- Decide at every assignment whether you want sharing or a copy, and make the copy explicit with `list(x)` when you do.
- Never assign the return value of `append`, `extend`, `sort`, `reverse`, `insert`, `remove` or `clear` - they return `None`.
- Prefer `sorted(x)` when you need the result; use `x.sort()` only when you deliberately want to reorder in place.
- Iterate a copy (`for v in list(items):`) when removing during iteration, or build a new list with a comprehension.
- Use `extend` or `+=` to splice many items; `append` is for exactly one item, including one list you mean to nest.
- Test membership repeatedly against a `set`, not a list - `in` on a list is O(n) per call.
- Return new lists from helpers; treat the caller's list as read-only unless the function name says it mutates.
- Reach for `collections.deque` when you genuinely queue from the front - `insert(0, x)` and `pop(0)` are O(n) by construction.

## 17. Performance Considerations

A list is an array of references, so its performance profile follows directly: random access and tail growth are cheap, front edits and membership scans are not. The table below is the version worth memorising - it explains every surprising slowdown in list-heavy code.

| Pattern | Cost | Preferred alternative |
| --- | --- | --- |
| `for _ in range(n): a.append(x)` | amortised O(n) | a comprehension builds it in C |
| `a.insert(0, x)` / `a.pop(0)` | O(n) each | `collections.deque` - O(1) at both ends |
| `x in a` inside a loop | O(n) per test | hoist into `seen = set(a)` once |
| `a = a + [x]` in a loop | O(n) per copy - quadratic | `a.append(x)` or `a += [x]` |
| `a[:]` of a huge list | O(n) always | share it read-only, or copy lazily |

In [ ]:
import time

n = 40_000

start = time.perf_counter()
acc = []
for i in range(n):
    acc = acc + [i]                 # reallocates every pass
concat_s = time.perf_counter() - start

start = time.perf_counter()
acc = []
for i in range(n):
    acc.append(i)                   # grows in place
append_s = time.perf_counter() - start

print(f'concat {concat_s:.4f}s vs append {append_s:.4f}s')
print(f'append is {concat_s / append_s:.1f}x faster here')

The difference is not a constant factor: `a = a + [x]` copies the whole accumulated list every pass, which is O(n^2) work overall, while `append` is amortised O(1) per call. For a ten-thousand-element log built the wrong way the gap is the difference between milliseconds and seconds.

## 18. Pythonic Approaches

Idiomatic list code reads as *what* rather than *how*: build with a comprehension instead of an empty list plus `append`, iterate the list rather than its indices, and let unpacking replace manual slicing.

In [ ]:
readings = [0.4, 0.9, 0.7]

# not pythonic
doubled = []
for value in readings:
    doubled.append(value * 2)

# pythonic - one expression, no manual bookkeeping
doubled = [value * 2 for value in readings]
print('doubled:', doubled)

first, *middle, last = readings
print('unpack :', first, middle, last)      # instead of [0], [1:-1], [-1]

- `for value in items:` not `for i in range(len(items)):` - the index is only needed when you use it.
- Build with comprehensions; reach for `append` inside a loop only when each iteration adds zero or one item conditionally.
- Use `any(...)` and `all(...)` for existence checks over a list - they short-circuit.
- `items += other` (in place) and `items + other` (new list) differ in both result identity and cost; pick deliberately.
- Sort with `key=` instead of pre-transforming the data.

## 19. Robotics Connection

A rover's waypoint list is shared by the planner, the logger and the safety monitor. If one component aliases the list and tidies it in place, the others silently see a different mission - which is exactly how a route loses a waypoint without any error being raised.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
plan = [(0.0, 0.0), (3.0, 0.0), (3.0, 4.0)]
logger_view = plan                 # alias, not a copy

for leg in range(2):
    robot.move(1.0, speed_mps=0.4)
    status = robot.status()
    logger_view.append(tuple(round(c, 2) for c in status['position']))

print('plan after drive:', plan)    # the plan GREW via the logger
print('battery:', round(robot.status()['battery_pct'], 1), '%')

The fix is a boundary copy: a component that only *reads* the plan gets `plan.copy()`, and a component that must publish a new plan returns a new list rather than editing the shared one. The rule - copy on the way in, return on the way out - is what keeps list sharing safe between modules.

## 20. Engineering Example

Batching a sensor stream into fixed-size chunks is a list routine that appears in every telemetry pipeline. Slicing by a computed stride keeps the loop index and the window size in one expression, and the final partial window is preserved rather than dropped.

In [ ]:
def batch(values, size):
    """Split values into consecutive chunks of at most size items."""
    if size <= 0:
        raise ValueError('size must be positive')
    return [values[i:i + size] for i in range(0, len(values), size)]

stream = [0.1, 0.2, 0.3, 0.4, 0.5]
print('batches of 2:', batch(stream, 2))
print('batches of 4:', batch(stream, 4))
print('source intact:', stream)

## 21. Guided Practice

1. Build a list with `append`, then print its length, first and last elements using positive and negative indexing.
2. Slice a window from a list of readings and prove the slice is a copy by mutating it and reprinting the source.
3. Reproduce the aliasing bug (`b = a; b.append(...)`) and then fix it with `list(a)`.
4. Sort a list of `(name, value)` tuples two ways: in place with a key, and into a new list with `sorted` - confirm which one changed the original.
5. Filter even numbers out of a list twice: by mutating during iteration (watch it fail) and with a comprehension (watch it work).

In [ ]:
readings = [0.4, 0.9, 0.7, 0.2]
print('len:', len(readings), ' first:', readings[0], ' last:', readings[-1])

window = readings[1:3]
window.append(9.9)
print('window:', window, ' source:', readings)   # copy proven

a, b = [1, 2], None
b = a
b.append(3)
print('aliasing made a =', a)

original = [3, 1, 2]
ordered = sorted(original)
print('sorted():', ordered, ' original:', original)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What does `b = a` do when `a` is a list?
- **Q1.** What is the cost of `xs.insert(0, value)` on a list of n elements?
- **Q1.** Which statement about `xs.copy()` on `xs = [[1], [2]]` is true?
- **Q1.** The rover's logger holds `plan = mission.plan` and then appends a waypoint. What is the safest fix?

## 23. Summary

A list is an ordered, mutable sequence backed by an array of references. That single sentence explains the performance table - O(1) indexing, amortised O(1) appends, O(n) front edits and membership scans - and it explains the two failure modes the topic is really about.

The first failure mode is **aliasing**: `b = a` creates a second name, not a second list, so a mutation through either name is seen by both. Copies must be requested explicitly, and every built-in copy is shallow. The second is **in-place mutation**: `append`, `extend`, `insert`, `remove`, `pop`, `sort`, `reverse` and `clear` all change the list and return `None`, so assigning their result destroys the data, and mutating during iteration skips elements.

The engineering discipline that follows is simple and universal: decide at every line whether you intend sharing or independence, copy deliberately at module boundaries, iterate a copy when you remove, and return new lists from helpers. Applied consistently, list bugs stop being silent - which, for a waypoint plan shared by three components of a rover, is the only acceptable outcome.

## 24. Key Takeaways

- Lists are ordered and mutable; assignment aliases, it never copies.
- `list(x)`, `x.copy()` and `x[:]` are shallow copies - inner objects are still shared.
- Mutating methods change the list in place and return `None`; `sorted(x)` returns a new list.
- `append` adds one element, `extend` splices many - confusing them changes the data's shape.
- `[x] * n` repeats references; use a comprehension for nested or mutable elements.
- Never remove from a list while iterating it directly - iterate a copy or build a new list.
- Indexing and tail appends are cheap; front edits and `x in a` are linear - use a `deque` or a `set` instead.
- Copy on the way into shared code, return new lists on the way out.

## 25. Further Exploration

- Read `collections.deque` in the standard library docs and note which list operations it makes O(1).
- Time `a = a + [x]` against `a.append(x)` for n = 10^4, 10^5 and 10^6, and confirm the quadratic curve.
- Investigate `copy.deepcopy` versus `copy.copy` on a list of dicts, and write the assertion that proves the difference.
- Look up `list.clear()`, `list.remove()` and `list.index()` edge cases - what does `remove` do when the value is absent?
- Preview `bisect.insort`: how does Python keep a sorted list balanced between insertion cost and search cost?